# Data Cleaning: Impacts of Data Centers on Electrical Utilities

This notebook loads the aggregated data from `01.5-data-aggregation.ipynb` saved to `data/processed/merged_data_centers.csv` and cleans it. The notebook also adds additional EIA data to contibute further features for data analysis.

In [14]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
import geocoder
from geopy.distance import geodesic

In [15]:
DATA_DIR = Path("data/raw")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

#### **Load Data Centers**

In [16]:
data_centers = pd.read_csv("./data/processed/merged_data_centers.csv")
print("Rows and columns:", data_centers.shape)
data_centers.sample(3)

Rows and columns: (1720, 76)


,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup,crypto_idx,date,name,address_crypto,city_crypto,state_crypto,lat_crypto,long_crypto,company,probable_power_source,mw_nytimes,nearest_dc_miles,nearest_dc_name,nearest_dc_operator,zip_crypto,atlas_idx,name_atlas,operator,facility_type,lat_atlas,long_atlas,capacity_operational_mw,capacity_planned_mw
447,3973870,8589a815-c978-485f-80da-9f0e1c33b48f,1770096246112-4255,TeraWulf Data Center,1627 state rte 3543,Hawesville,KY,42348.0,Hancock,37.941345,-86.78807,Proposed,High,Google Maps and other sources,NaN,TeraWulf,Anthropic,480,480.0,480.0,Hyperscale (100-999 MW),5,Grid (unspecified mix),NaN,NaN,NaN,NaN,NaN,NaN,250.0,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://investors.terawulf.com/news-events/pre...,https://seekingalpha.com/news/4545877-terawulf...,https://www.cnbc.com/2026/07/06/anthropic-tera...,NaN,NaN,NaN,NaN,NaN,1.770077e+12,1.783296e+12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
496,3973919,31b7a93c-f6de-4f1b-ab6b-80317c8a7003,1782942807947-6996,Loring Commerce Center Data center,near georgia rd,Limestone,ME,4750.0,Aroostook,46.949330,-67.89270,Proposed,Medium,Google Maps and other sources,NaN,NaN,NaN,6,6.0,50.0,Medium (11-50 MW),3,"Hydroelectric, Grid (unspecified mix)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://insideclimatenews.org/news/26112025/ma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.782864e+12,1.782864e+12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1165,3974588,68414db6-d8bc-42b6-a10b-38b14c97fd28,1785258427594-3902,Diode Ventures Data Center,hwy 274 key ranch rd,Trinidad,TX,75163.0,Henderson,32.205753,-96.11986,Cancelled,High,Google Maps only,NaN,Diode Ventures,NaN,150,80.0,150.0,Hyperscale (100-999 MW),5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,NaN,Organized Advocacy,NaN,Our Lake Our Voice,NaN,https://savecedarcreeklake.com/take-action.html,NaN,KE,Media Monitoring,https://www.msn.com/en-us/news/us/plans-scrapp...,https://www.msn.com/en-us/news/us/the-fight-is...,https://www.texastribune.org/2026/07/23/east-t...,https://www.datacenterdynamics.com/en/news/dio...,NaN,NaN,NaN,NaN,1.785197e+12,1.785197e+12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Summary Statistics for Data Centers Dataset
# Looking for anything that needs to be cleaned
data_centers.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
OBJECTID,1720.0,NaN,NaN,NaN,3974282.5,496.665548,3973423.0,3973852.75,3974282.5,3974712.25,3975142.0
GlobalID,1720,1720,3d8a6ce3-5e4b-4ede-b9ca-c9ab4f5dfa9b,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_id,1720,1720,1779305646902-5353,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_name,1720,1593,H5 Data Center,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,1675,1649,farmtomarket 157,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,1716,847,Sterling,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,1720,49,VA,464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip,1707.0,NaN,NaN,NaN,43942.100762,26479.577751,1085.0,20166.0,30349.0,72933.0,99734.0
county,1717,481,Loudoun,192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,1720.0,NaN,NaN,NaN,37.588261,4.236366,21.480452,33.778053,38.746981,40.060205,70.18478


#### **Missing Values**

After joining the datasets together we can use the new features to updated existing features where values are missing. The following updates instances where purpose, power_source, and mw are blank in the FracTracker dataset but exists in another dataset.

In [18]:
def update_missing_rows(row):
    # Assign missing purpose
    if pd.isna(row["purpose"]) and pd.notna(row["facility_type"]):
        row["purpose"] = row["facility_type"]

    # Assign missing mw
    if pd.isna(row["mw"]):
        if pd.notna(row["mw_nytimes"]):
            row["mw"] = row["mw_nytimes"]
        elif pd.notna(row["capacity_operational_mw"]):
            row["mw"] = row["capacity_operational_mw"]
        elif pd.notna(row["capacity_planned_mw"]):
            row["mw"] = row["capacity_planned_mw"]

    # Assign missing power source
    if (
        pd.isna(row["power_source"])
        and pd.notna(row["probable_power_source"])
        and row["probable_power_source"] != "Unknown"
    ):
        row["power_source"] = row["probable_power_source"]

    return row


# Use datasets to fill in missing values
data_centers = data_centers.apply(update_missing_rows, axis=1)

### **Data Centers - Data Cleaning**

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `sizerank` / `sizerank_numeric` | Mismatched values that are not consistent across categories. | Align sizerank wiht sizerank_numeric column labels. |
| `mw` | Contains both numerical values and ranges. | Separate ranges into `mw_low` and `mw_high` to preserve the available information. |
| `power_source` | String categorical values. | Standardize categories and encode them for modeling. |
| `dedicated_power_plant` | Limited availability across facilities. | Determine if power plants can be identified using location (latitude/longitude) and power source. |
| `number_of_generators` | Contains numerical information stored as strings; limited availability across facilities. | Too limited to be a learnable feature unless more information can be added. |
| `number_of_buildings` | Contains integers, ranges, and strings (e.g., `6+`, `1-19`, `8`, `3 large, 1 small`). | Create a consistent approach for representing ranges and descriptive counts. |
| `cooling_source` | Categorical values stored as strings. | Standardize categories and encode them for modeling. |
| `cooling_type` | Inconsistent categorical formatting (e.g., `Closed loop` vs. `Closed Loop`). | Normalize categorical values and encode them for modeling. |
| `project_cost` | Contains strings and potentially ranges rather than consistent numerical values. | Extract numerical values and determine an appropriate method for representing ranges. |



`mw`

In [19]:
# Function for standardizing MW values
def standardize_mw(value):
    if pd.isna(value):
        return np.nan, np.nan

    text = str(value).lower().strip()

    # Remove commas and fix typos
    text = text.replace(",", "").replace("nw", "mw")

    # Extract numbers and their units (MW or GW)
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(mw|gw)?', text)

    if not matches:
        return np.nan, np.nan

    values_mw = []

    for number, unit in matches:
        number = float(number)

        # Convert GW to MW
        if unit == "gw":
            number *= 1000

        values_mw.append(number)

    return min(values_mw), max(values_mw)


# Apply to the merged dataset
data_centers[["mw_low", "mw_high"]] = pd.DataFrame(
    data_centers["mw"].apply(standardize_mw).tolist(),
    index=data_centers.index
)

# Review cleaned results
data_centers.loc[data_centers["mw"].notna(),["mw", "mw_low", "mw_high"]].head(10)

,mw,mw_low,mw_high
0,3000,3000.0,3000.0
1,120,120.0,120.0
2,1200,1200.0,1200.0
3,300,300.0,300.0
4,5.0,5.0,5.0
6,210,210.0,210.0
7,500,500.0,500.0
8,13.5,13.5,13.5
10,100-200,100.0,200.0
12,8,8.0,8.0


`sizerank` / `sizerank_numeric`

In [20]:
# Mappings
size_rank_mw = {
   'Unknown': 1,
   'Small (0-10 MW)': 2,
   'Medium (11-50 MW)': 3,
   'Large (51-99 MW)': 4,
   'Hyperscale (100-999 MW)': 5,
   'Mega campus (>1,000 MW)': 6
}

# Fix sizerank labels
data_centers['sizerank'] = data_centers['sizerank'].replace('Mega Campus (>1,000 MW)', 'Mega campus (>1,000 MW)')

# Fix sizerank_numeric values to match sizerank
data_centers['sizerank_numeric'] = data_centers['sizerank'].apply(lambda x: size_rank_mw[x])



`power_source`

In [21]:
# Reuse power sources from FracTracker data centers
power_sources = [
    "batteries", "biomass", "coal", "fuel_cells",
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid",
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel",
    "renewable", "solar", "wind"
]

# Mapping given sources to predefined power sources
def normalize_sources(source):
    if pd.isna(source):
        return []

    found_sources = set()

    # Keyword mappings
    keyword_map = {
        "batter": "batteries",
        "biomass": "biomass",
        "coal": "coal",
        "fuel cell": "fuel_cells",
        "geothermal": "geothermal",
        "hydro": "hydroelectric",
        "gas": "natural_gas",
        "nuclear": "nuclear",
        "diesel": "oil_diesel",
        "oil": "oil_diesel",
        "solar": "solar",
        "wind": "wind",
        "renewable": "renewable",
        "wte": "renewable",
    }

    if "hybrid" in source:
        found_sources.add("hybrid_onsite_and_grid")
        source = source.replace("hybrid (onsite and grid)", "")
    
    for keyword, category in keyword_map.items():
        if keyword in source:
            found_sources.add(category)
            
    if "grid" in source:
        found_sources.add("grid_unspecified_mix")

    return sorted(found_sources)

# Map raw text to the standard categories
power_lists = data_centers["power_source"].str.lower().apply(normalize_sources)

# One 0/1 column per category
for src in power_sources:
    data_centers[src] = power_lists.apply(lambda found: int(src in found))

columns = ["power_source"] + power_sources

data_centers.loc[data_centers["power_source"].notna(), columns].sample(5)

,power_source,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind
1089,Natural gas,0,0,0,0,0,0,0,0,1,0,0,0,0,0
447,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
913,Natural gas,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1121,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
938,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0


`number_of_buildings`

In [22]:
# Standardization function for assigning number of buildings
def standardize_buildings(buildings):
    if pd.isna(buildings):
        return np.nan, np.nan
    
    buildings = buildings.replace(' ', '')
    b_low = np.nan
    b_high = np.nan
    
    if '-' in buildings:
        split = buildings.split('-')
        b_low = int(split[0])
        b_high = int(split[1])
        
    elif '+' in buildings:
        buildings = buildings.replace('+', '')
        b_low = int(buildings)
        
    elif buildings.isdigit():
        b_low = int(buildings)
        b_high = int(buildings)
        
    return b_low, b_high
    
# Add new features to handle ranges
data_centers[["buildings_low", "buildings_high"]] = pd.DataFrame(
    data_centers["number_of_buildings"].apply(standardize_buildings).tolist(),
    index=data_centers.index
)

data_centers.loc[data_centers["buildings_low"].notna()][["number_of_buildings", "buildings_low", "buildings_high"]].head()

,number_of_buildings,buildings_low,buildings_high
2,18,18.0,18.0
10,4,4.0,4.0
11,2,2.0,2.0
15,14,14.0,14.0
16,4,4.0,4.0


`cooling_source` + `cooling_type`

In [23]:
for col in ['cooling_source', 'cooling_type']:
    # Normalize cooling cols
    cooling_data = (
        data_centers[col]
        .str.lower()
        .str.strip()
        .str.replace('/', '_')
        .str.replace(' ', '_')
    )

    # One-hot encoding for each source type
    source_encoded = cooling_data.str.get_dummies()
    data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = ['cooling_source', 'air', 'water', 'hybrid_air_water']

data_centers.loc[data_centers['cooling_source'].notna()][columns].sample(5)

,cooling_source,air,water,hybrid_air_water
1667,Water,0,1,0
350,Air,1,0,0
868,Water,0,1,0
1189,Water,0,1,0
115,Water,0,1,0


In [24]:
columns = ['cooling_type', 'closed_loop', 'fans', 'open_loop']

data_centers.loc[data_centers['cooling_type'].notna()][columns].sample(5)

,cooling_type,closed_loop,fans,open_loop
699,Closed loop,1,0,0
1061,Closed loop,1,0,0
97,Closed loop,1,0,0
618,Closed loop,1,0,0
105,Closed loop,1,0,0


`project_cost`

In [25]:
def standardize_project_cost(cost):
    if pd.isna(cost):
        return np.nan, np.nan
    
    multiplier = 1
    cost_low = np.nan
    cost_high = np.nan
    
    # Normalize cost
    cost = (
        str(cost)
        .lower()
        .replace(' ', '')
        .replace('$', '')
        .replace(',', '')
        .replace('miillion', 'million')
    )

    # Drop notes in parentheses
    cost = re.sub(r'\(.*?\)', '', cost)

    # Replace abbreviated m
    cost = re.sub(r'm$', 'million', cost)
    
    if 'million' in cost:
        multiplier = 1000000
    elif 'billion' in cost:
        multiplier = 1000000000
        
    cost = re.sub(r'[mb]illion', '', cost)
    
    # Assign range to low and high
    if '-' in cost:
        split = cost.split('-')
        cost_low = float(split[0]) * multiplier
        cost_high = float(split[1]) * multiplier
        
    elif '+' in cost:
        cost = cost.replace('+', '')
        cost_low = float(cost) * multiplier
        
    else:
        cost_low = float(cost) * multiplier
        cost_high = float(cost) * multiplier
        
    return cost_low, cost_high

# Add new features for handling the cost range
data_centers[["project_cost_low", "project_cost_high"]] = pd.DataFrame(
    data_centers["project_cost"].apply(standardize_project_cost).tolist(),
    index=data_centers.index
)

data_centers[data_centers["project_cost"].notna()][["project_cost", "project_cost_low", "project_cost_high"]].sample(5)

,project_cost,project_cost_low,project_cost_high
286,5.7 million,5.700000e+06,5.700000e+06
249,$300 million,3.000000e+08,3.000000e+08
1123,$56 million,5.600000e+07,5.600000e+07
204,$2.4 billion,2.400000e+09,2.400000e+09
237,$1.2 billion,1.200000e+09,1.200000e+09


#### **Adding 'is_crypto' label for the aggregated dataset**

**Method 1:** Using the aggregated fields ('crypto_idx') from the crypto dataset we can determine crypto rows. <br>
**Method 2:** Using the purpse and name fields we can identify additional crypto facilities.

In [26]:
# "^crypto" means the text starts with "crypto"
crypto_words = "bitcoin|crypto mining|crypto mine|cryptomine|cryptocurrency|^crypto"

# Method 1 
# 'crypto_idx' is a crypto column and will appear where the datasets were merged
crypto_from_crypto_dataset = data_centers['crypto_idx'].notna()

# Method 2
purpose_says_crypto = data_centers["purpose"].str.contains(crypto_words, case=False, na=False)
name_says_crypto = data_centers["facility_name"].str.contains(crypto_words, case=False, na=False)

crypto_from_name_purpose = purpose_says_crypto | name_says_crypto

print("Crypto from crypto dataset fields after merge:", crypto_from_crypto_dataset.sum())
print("Crypto from purpose or name:", crypto_from_name_purpose.sum())

Crypto from crypto dataset fields after merge: 29
Crypto from purpose or name: 23


##### **Final label**

A data center gets `is_crypto = 1` if any of the three checks is True.

In [27]:
is_crypto = crypto_from_crypto_dataset | crypto_from_name_purpose

data_centers['is_crypto'] = is_crypto.astype(int)

print("Data centers labeled crypto:", data_centers["is_crypto"].sum(), "of", len(data_centers))
print("Share labeled crypto:", round(data_centers["is_crypto"].mean() * 100, 1), "%")

Data centers labeled crypto: 47 of 1720
Share labeled crypto: 2.7 %


### EIA-860 Plant Data

The first row of each EIA-860 Excel file is a title, so the column names are on the second row (`header=1`).

In [28]:
# Added header=1 to skip the first row

plants = pd.read_excel(DATA_DIR / "eia860_2025" / "2___Plant_Y2025.xlsx", header=1)
print("Rows and columns:", plants.shape)
plants.sample(3)

Rows and columns: (17349, 42)


,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,Longitude,NERC Region,Balancing Authority Code,Balancing Authority Name,Name of Water Source,Primary Purpose (NAICS Code),Regulatory Status,Sector,Sector Name,FERC Cogeneration Status,FERC Cogeneration Docket Number,FERC Small Power Producer Status,FERC Small Power Producer Docket Number,FERC Exempt Wholesale Generator Status,FERC Exempt Wholesale Generator Docket Number,Ash Impoundment?,Ash Impoundment Lined?,Ash Impoundment Status,Transmission or Distribution System Owner,Transmission or Distribution System Owner ID,Transmission or Distribution System Owner State,Grid Voltage (kV),Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
5375,17609,Southern California Edison Co,57235,Solar Photovoltaic Project #21,13053 San Bernardino Ave,Fontana,CA,92335,San Bernardino,34.075278,-117.553611,WECC,CISO,California Independent System Operator,NaN,22,RE,1,Electric Utility,N,NaN,N,NaN,N,NaN,NaN,X,NaN,Southern California Edison Co,17609,CA,12,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6569,58594,Rockford Solar Partners,58643,Rockford Solar Farm,7201 Beltline Drive,Rockford,IL,61109,Winnebago,42.175278,-89.088333,RFC,PJM,"PJM Interconnection, LLC",NaN,22,NR,2,IPP Non-CHP,N,NaN,N,NaN,N,NaN,N,X,NaN,Commonwealth Edison Co,4110,IL,12.47,,,N,NaN,NaN,NaN,NaN,NaN,X,X
17288,68008,"Belvidere Landfill Solar 2 East, LLC",70390,IL - Ameresco - Belvidere East,9835 IL Ste 76,Belvidere,IL,61008,Boone,42.295704,-88.849153,MRO,PJM,"PJM Interconnection, LLC",NaN,22,NR,2,IPP Non-CHP,N,NaN,Y,25-768,N,NaN,NaN,NaN,NaN,Commonwealth Edison Co,4110,IL,12.47,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
plants.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,17349.0,NaN,NaN,NaN,47106.087844,23529.704706,7.0,18642.0,60301.0,64545.0,68045.0
Utility Name,17349,6935,"Altus Power America Management, LLC",247,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,17349.0,NaN,NaN,NaN,52099.079601,22868.185984,1.0,55701.0,61103.0,65921.0,70452.0
Plant Name,17349,17345,Viola,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Street Address,17229,15510,TBD,789,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,17314,6015,Lancaster,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,17349,51,CA,2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip,17349.0,8431.0,93501.0,64.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,17326,1542,Kern,304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,17349,16666,,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### EIA-860 Generator Data

The generator file has three sheets: `Operable`, `Proposed`, and `Retired and Canceled`. For now we only look at `Operable`.

In [30]:
generators = pd.read_excel(
    DATA_DIR / "eia860_2025" / "3_1_Generator_Y2025.xlsx",
    sheet_name="Operable",
    header=1
)
print("Rows and columns:", generators.shape)
generators.sample(3)

Rows and columns: (27919, 73)


,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,Ownership,Duct Burners,Can Bypass Heat Recovery Steam Generator?,RTO/ISO LMP Node Designation,RTO/ISO Location Designation for Reporting Wholesale Sales Data to FERC,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Uprate or Derate Completed During Year,Month Uprate or Derate Completed,Year Uprate or Derate Completed,Status,Synchronized to Transmission Grid,Operating Month,Operating Year,Planned Retirement Month,Planned Retirement Year,Associated with Combined Heat and Power System,Sector Name,Sector,Topping or Bottoming,Energy Source 1,Energy Source 2,Energy Source 3,Energy Source 4,Energy Source 5,Energy Source 6,Startup Source 1,Startup Source 2,Startup Source 3,Startup Source 4,Solid Fuel Gasification System?,Carbon Capture Technology?,Turbines or Hydrokinetic Buoys,Time from Cold Shutdown to Full Load,Fluidized Bed Technology?,Pulverized Coal Technology?,Stoker Technology?,Other Combustion Technology?,Subcritical Technology?,Supercritical Technology?,Ultrasupercritical Technology?,Planned Net Summer Capacity Uprate (MW),Planned Net Winter Capacity Uprate (MW),Planned Uprate Month,Planned Uprate Year,Planned Net Summer Capacity Derate (MW),Planned Net Winter Capacity Derate (MW),Planned Derate Month,Planned Derate Year,Planned New Prime Mover,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
2846,18895,Thumb Electric Coop of Mich,1875.0,Ubly,MI,Huron,4,Petroleum Liquids,IC,NaN,S,X,X,NaN,NaN,0.9,0.8,0.9,0.9,0.5,N,,,OP,X,8.0,1947.0,,,N,Electric Utility,1.0,X,DFO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,1H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,NaN,N
237,13870,"Nushagak Electric Coop, Inc",109.0,Dillingham,AK,Dillingham,14,Petroleum Liquids,IC,NaN,S,X,X,NaN,NaN,1.0,0.8,1,1,0.5,N,,,OP,X,3.0,2008.0,,,N,Electric Utility,1.0,X,DFO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,,10M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,N,N
21878,63229,Essex County Correctional Facility Cogen,63540.0,Essex County Correctional Facility Cogen,NJ,Essex,EG1,Natural Gas Internal Combustion Engine,IC,NaN,W,X,X,NaN,NaN,3.1,0.8,3,3,1.5,N,,,OP,X,1.0,2003.0,,,Y,IPP CHP,3.0,T,NG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,1H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,NaN,NaN


In [31]:
generators.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,27919.0,5853.0,55938.0,329.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Utility Name,27918,5842,Industrial Power Generating Company LLC,329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,27918.0,NaN,NaN,NaN,42375.107386,26782.287436,1.0,7146.0,56794.5,62643.75,70438.0
Plant Name,27918,14325,Edison Sault,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,27918,51,CA,3161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,27918,1479,Los Angeles,424,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Generator ID,27917,10890,1,2923,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Technology,27918,27,Solar Photovoltaic,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Prime Mover,27918,18,PV,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unit Code,2180,310,CC1,770,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Export data prior to failed cells

In [32]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")

Saved: fractracker_data_centers_clean.csv with 1720 rows and 101 columns


EIA-860 Defining Energy Source Features

In [33]:
energy_sources = pd.read_csv(
    DATA_DIR / "eia860_2025" / "eia_860_energy_source_codes.csv",
)
energy_sources.rename(columns={'Energy Source Code': 'Energy Source 1'}, inplace=True)
energy_sources.sample(3)

,Energy Source 1,Description
6,RC,Refined Coal
17,H2,Hydrogen
21,OBS,Other Biomass Solids (specify in SCHEDULE 7)


In [34]:
generators['Energy Source 1'].unique()
generator_features = generators[['State','County','Energy Source 1','Technology','Time from Cold Shutdown to Full Load','Summer Capacity (MW)','Winter Capacity (MW)']].copy()
# Dropping description to create categorical feature from Energy Source
generator_features_full = pd.merge(generator_features, energy_sources, on='Energy Source 1', how='left').drop('Description', axis=1)
generator_features_full.rename(columns={'Category':'Energy Category','Time from Cold Shutdown to Full Load':'Time to Full Load'}, inplace=True)
# Clean Time to Full Load from string to numerical value
generator_features_full.sample(15)

,State,County,Energy Source 1,Technology,Time to Full Load,Summer Capacity (MW),Winter Capacity (MW)
12835,RI,Providence,LFG,Landfill Gas,1H,1.5,1.5
10026,TX,Jasper,WAT,Conventional Hydroelectric,10M,4,4
15318,GA,Walton,NG,Natural Gas Fired Combustion Turbine,1H,149.9,172
13058,VA,Bedford,WAT,Conventional Hydroelectric,12H,0.2,0.2
11907,NY,Jefferson,WAT,Conventional Hydroelectric,1H,1.2,1.2
25906,MA,Worcester,SUN,Solar Photovoltaic,NaN,2,2
18642,WA,King,LFG,Landfill Gas,1H,0.3,0.3
16486,IL,Winnebago,NG,Natural Gas Fired Combustion Turbine,1H,180.7,188
1581,FL,Pinellas,DFO,Petroleum Liquids,10M,37,55
3923,MI,St Joseph,WAT,Conventional Hydroelectric,1H,0.4,0.4


Time to Full Load in Minutes Clean Up

In [35]:
def time_in_min(stringval):
    val = str(stringval).lower().strip()
    # Match digits versus text
    regex = re.match(r"(\d+)(\s*[a-z]+)", val)
    if not regex:
        return None

    if regex.group(2) in ['m']:
        return int(regex.group(1))
    elif regex.group(2) in ['h']:
        return int(regex.group(1))*60
    else:
        return None

In [36]:
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].apply(time_in_min)


In [37]:
generator_features_full.describe()
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].fillna(generator_features_full['Time to Full Load'].mean().round(1))

One-Hot Encoding for Energy

In [38]:
one_hot_energy = pd.get_dummies(generator_features_full['Technology']).astype(int)
generator_features_full = generator_features_full.join(one_hot_energy)

#convert from object to numeric for MW
generator_features_full[["Summer Capacity (MW)", "Winter Capacity (MW)"]] = generator_features_full[["Summer Capacity (MW)", "Winter Capacity (MW)"]].apply(
    pd.to_numeric,
    errors = "coerce"
)

In [39]:
generator_features_full.sample(5)

,State,County,Energy Source 1,Technology,Time to Full Load,Summer Capacity (MW),Winter Capacity (MW),All Other,Batteries,Coal Integrated Gasification Combined Cycle,Conventional Hydroelectric,Conventional Steam Coal,Flywheels,Geothermal,Hydroelectric Pumped Storage,Landfill Gas,Municipal Solid Waste,Natural Gas Fired Combined Cycle,Natural Gas Fired Combustion Turbine,Natural Gas Internal Combustion Engine,Natural Gas Steam Turbine,Natural Gas with Compressed Air Storage,Nuclear,Offshore Wind Turbine,Onshore Wind Turbine,Other Gases,Other Natural Gas,Other Waste Biomass,Petroleum Coke,Petroleum Liquids,Solar Photovoltaic,Solar Thermal with Energy Storage,Solar Thermal without Energy Storage,Wood/Wood Waste Biomass
7114,TX,Frio,NG,Natural Gas Internal Combustion Engine,10.0,8.4,8.4,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6822,TN,Marion,WAT,Conventional Hydroelectric,60.0,25.0,25.6,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
28385,TX,Hidalgo,DFO,Petroleum Liquids,10.0,0.5,0.6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
13341,PA,Allegheny,OG,Other Gases,60.0,8.0,8.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
19585,MD,Prince Georges,SUN,Solar Photovoltaic,140.7,1.0,1.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


In [40]:
generator_features_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 32050 entries, 0 to 32049
Data columns (total 34 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   State                                        32049 non-null  str    
 1   County                                       32049 non-null  str    
 2   Energy Source 1                              32049 non-null  str    
 3   Technology                                   32049 non-null  str    
 4   Time to Full Load                            32050 non-null  float64
 5   Summer Capacity (MW)                         31964 non-null  float64
 6   Winter Capacity (MW)                         31938 non-null  float64
 7   All Other                                    32050 non-null  int64  
 8   Batteries                                    32050 non-null  int64  
 9   Coal Integrated Gasification Combined Cycle  32050 non-null  int64  
 10  Conventio

In [41]:
non_energy = {
    "State",
    "County",
    "Energy Source 1",
    "Technology",
    "Time to Full Load",
    "Summer Capacity (MW)",
    "Winter Capacity (MW)"
}

energy_source = [
    energytech for energytech in generator_features_full.columns
    if energytech not in non_energy
]

group_by_stat = generator_features_full.groupby(["State","County"], as_index = False).agg(
    generator_count = ("County","size"),
    generator_summer_max = ("Summer Capacity (MW)","max"),
    generator_winter_max = ("Winter Capacity (MW)","max"),
    generator_capacity_summer = ("Summer Capacity (MW)","sum"),
    generator_capacity_winter = ("Winter Capacity (MW)","sum"),
    generator_avg_time_to_full = ("Time to Full Load","mean"),
    **{
        source:(source,"max")
        for source in energy_source
    }
)
group_by_stat["Capacity_Sum"] = group_by_stat['generator_capacity_summer'] + group_by_stat['generator_capacity_winter']
group_by_stat.sample(5)

,State,County,generator_count,generator_summer_max,generator_winter_max,generator_capacity_summer,generator_capacity_winter,generator_avg_time_to_full,All Other,Batteries,Coal Integrated Gasification Combined Cycle,Conventional Hydroelectric,Conventional Steam Coal,Flywheels,Geothermal,Hydroelectric Pumped Storage,Landfill Gas,Municipal Solid Waste,Natural Gas Fired Combined Cycle,Natural Gas Fired Combustion Turbine,Natural Gas Internal Combustion Engine,Natural Gas Steam Turbine,Natural Gas with Compressed Air Storage,Nuclear,Offshore Wind Turbine,Onshore Wind Turbine,Other Gases,Other Natural Gas,Other Waste Biomass,Petroleum Coke,Petroleum Liquids,Solar Photovoltaic,Solar Thermal with Energy Storage,Solar Thermal without Energy Storage,Wood/Wood Waste Biomass,Capacity_Sum
1200,NC,Clay,8,13.9,13.1,31.40,29.80,22.500000,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,61.20
1298,ND,Stutsman,4,200.0,200.0,335.49,347.69,220.175000,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,683.18
90,AR,Hot Spring,13,559.2,633.9,1745.50,1880.80,347.746154,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,3626.30
2257,WI,Washington,11,79.8,101.8,313.50,396.80,74.672727,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,710.30
695,KS,Clark,8,200.0,200.0,603.70,603.70,90.262500,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1207.40


In [42]:
group_by_stat.to_csv("data/processed/EIA860_Generator_Cleaned.csv", index=False)

### EIA-861M Retail Sales of Electricity


In [43]:
# Added header=2 to skip the first two rows of the Excel sheet

sales = pd.read_excel(DATA_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
print("Rows and columns:", sales.shape)
sales.sample(3)

Rows and columns: (10150, 24)


,Year,Month,State,Data Status,Thousand Dollars,Megawatthours,Count,Cents/kWh,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1,Thousand Dollars.2,Megawatthours.2,Count.2,Cents/kWh.2,Thousand Dollars.3,Megawatthours.3,Count.3,Cents/kWh.3,Thousand Dollars.4,Megawatthours.4,Count.4,Cents/kWh.4
8894,2012,1.0,MD,Final,330087.00,2644386.0,2212862.0,12.48,266607.00,2499521.0,246564.0,10.67,32681.00,396851.0,8816.0,8.24,3319.00,47602.00,5.0,6.97,632694.00,5588359.0,2468247.0,11.32
586,2025,8.0,MS,Preliminary,284838.55,2095981.8,1349345.0,13.59,187122.18,1467419.2,248094.0,12.75,109644.56,1464263.2,12911.0,7.49,0.00,0.00,0.0,0.00,581605.29,5027664.2,1610350.0,11.57
1105,2024,10.0,NY,Final,860703.66,3435335.8,7442248.0,25.05,1063505.30,5583606.0,1166506.0,19.05,121971.05,1288102.3,16511.0,9.47,29069.47,218656.73,6.0,13.29,2075249.50,10525701.0,8625271.0,19.72


In [44]:
sales.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Year,10150.0,18.0,2025.0,612.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Month,10149.0,NaN,NaN,NaN,6.41206,3.442597,1.0,3.0,6.0,9.0,12.0
State,10149,51,AK,199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Status,10149,2,Final,9180,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Thousand Dollars,10149.0,NaN,NaN,NaN,322406.257117,374363.877149,13064.0,78291.42,218232.0,404119.32,3266301.5
Megawatthours,10149.0,NaN,NaN,NaN,2358136.574637,2557344.034097,104574.0,595182.0,1706973.5,3135968.7,22481421.0
Count,10149.0,NaN,NaN,NaN,2631403.392847,2731937.969772,226548.0,802281.0,1971280.0,3106777.0,14863374.0
Cents/kWh,10149.0,NaN,NaN,NaN,14.26151,5.25453,6.73,11.08,12.7,15.63,52.72
Thousand Dollars.1,10149.0,NaN,NaN,NaN,251960.525962,316264.755862,20778.0,66788.65,166221.67,313892.57,3327766.4
Megawatthours.1,10149.0,NaN,NaN,NaN,2241268.764867,2416984.780473,126343.74,642345.04,1623061.3,2826659.0,17843545.0


In [45]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 10150 entries, 0 to 10149
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Year                10150 non-null  object 
 1   Month               10149 non-null  float64
 2   State               10149 non-null  str    
 3   Data Status         10149 non-null  str    
 4   Thousand Dollars    10149 non-null  float64
 5   Megawatthours       10149 non-null  float64
 6   Count               10149 non-null  float64
 7   Cents/kWh           10149 non-null  float64
 8   Thousand Dollars.1  10149 non-null  float64
 9   Megawatthours.1     10149 non-null  float64
 10  Count.1             10149 non-null  float64
 11  Cents/kWh.1         10149 non-null  float64
 12  Thousand Dollars.2  10149 non-null  float64
 13  Megawatthours.2     10149 non-null  float64
 14  Count.2             10149 non-null  float64
 15  Cents/kWh.2         10149 non-null  float64
 16  Thousand Dollar

In [46]:
sales['Year'] = sales['Year'].astype(str)

In [47]:
# Looking at 2025 only and only .1 since that is Commercial
# EIA classifies data centers as commercial customers
sales_2025 = sales[sales['Year']=='2025'].copy()
sales_2025 = sales_2025[['Year','Month','State','Thousand Dollars.1','Megawatthours.1','Count.1','Cents/kWh.1']]
sales_2025.sample(5)

,Year,Month,State,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1
701,2025,6.0,PA,416688.73,3369799.80,742554.0,12.37
433,2025,11.0,MS,140072.99,1036234.20,248261.0,13.52
680,2025,6.0,KY,222790.47,1894947.50,327745.0,11.76
926,2025,1.0,DE,46416.99,372939.93,60610.0,12.45
852,2025,3.0,OK,160386.63,1975252.70,310680.0,8.12


In [48]:
# Group by State, Summer/Winter AVG
# Summer/Winter to match generator dataset capacity values
sales_2025['Season'] = sales_2025['Month'].apply(
    lambda x: 'summer' if 4 <= x <= 9 else 'winter'
)
grouped_sales = sales_2025.groupby(by=['State','Season'], as_index=False).mean(numeric_only=True).sort_values(by='State', ascending=True).drop(columns=['Month'])
grouped_sales.head(10)

,State,Season,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1
0,AK,summer,4.818776e+04,2.120679e+05,5.840083e+04,22.718333
1,AK,winter,5.055212e+04,2.303088e+05,5.806733e+04,21.958333
2,AL,summer,2.980218e+05,2.057032e+06,3.959620e+05,14.486667
3,AL,winter,2.590876e+05,1.779264e+06,3.959318e+05,14.563333
4,AR,summer,1.166311e+05,1.070972e+06,2.092887e+05,10.906667
5,AR,winter,9.688299e+04,9.132678e+05,2.091840e+05,10.611667
6,AZ,summer,4.453811e+05,3.455614e+06,3.505910e+05,12.855000
7,AZ,winter,3.346111e+05,2.801567e+06,3.505335e+05,11.940000
8,CA,summer,2.682709e+06,9.741230e+06,1.776620e+06,27.353333
9,CA,winter,2.304272e+06,9.178307e+06,1.766771e+06,25.031667


In [49]:
# Pivot and Flatten

pivot = (
    grouped_sales.pivot(
        index = 'State',
        columns = 'Season',
        values = [
            'Thousand Dollars.1',
            'Megawatthours.1',
            'Count.1',
            'Cents/kWh.1'
            ]
    )
)

pivot.columns = [
    f'{season}_{metric}'
    for metric, season in pivot.columns
    ]

pivot_sales = pivot.reset_index()
pivot_sales.head(20)


,State,summer_Thousand Dollars.1,winter_Thousand Dollars.1,summer_Megawatthours.1,winter_Megawatthours.1,summer_Count.1,winter_Count.1,summer_Cents/kWh.1,winter_Cents/kWh.1
0,AK,4.818776e+04,5.055212e+04,2.120679e+05,2.303088e+05,5.840083e+04,5.806733e+04,22.718333,21.958333
1,AL,2.980218e+05,2.590876e+05,2.057032e+06,1.779264e+06,3.959620e+05,3.959318e+05,14.486667,14.563333
2,AR,1.166311e+05,9.688299e+04,1.070972e+06,9.132678e+05,2.092887e+05,2.091840e+05,10.906667,10.611667
3,AZ,4.453811e+05,3.346111e+05,3.455614e+06,2.801567e+06,3.505910e+05,3.505335e+05,12.855000,11.940000
4,CA,2.682709e+06,2.304272e+06,9.741230e+06,9.178307e+06,1.776620e+06,1.766771e+06,27.353333,25.031667
5,CO,2.322718e+05,2.006505e+05,1.779203e+06,1.693516e+06,4.017005e+05,4.015098e+05,13.028333,11.851667
6,CT,2.252811e+05,2.169308e+05,9.864365e+05,9.270143e+05,1.563380e+05,1.565435e+05,22.905000,23.406667
7,DC,1.251229e+05,1.205704e+05,6.262489e+05,5.772834e+05,2.759117e+04,2.761783e+04,19.915000,20.900000
8,DE,4.709125e+04,4.300572e+04,3.754953e+05,3.374110e+05,6.092983e+04,6.083633e+04,12.545000,12.751667
9,FL,1.039294e+06,8.833459e+05,9.121830e+06,7.636917e+06,1.324915e+06,1.324531e+06,11.398333,11.565000


In [50]:
pivot_sales.to_csv("data/processed/EIA861M_Sales_cleaned.csv", index=False)

### EIA-930 Balance Data

In [51]:
# This is a subset of the EIA 930 Balance dataset for the second half of 2025

balance = pd.read_csv(DATA_DIR / "eia930" / "EIA930_BALANCE_2025_Jul_Dec.csv")
print("Rows and columns:", balance.shape)
balance.sample(3)

Rows and columns: (267218, 65)


,Balancing Authority,Data Date,Hour Number,Local Time at End of Hour,UTC Time at End of Hour,Demand Forecast (MW),Demand (MW),Net Generation (MW),Total Interchange (MW),Sum(Valid DIBAs) (MW),Demand (MW) (Imputed),Net Generation (MW) (Imputed),Total Interchange (MW) (Imputed),Demand (MW) (Adjusted),Net Generation (MW) (Adjusted),Total Interchange (MW) (Adjusted),Net Generation (MW) from Coal,Net Generation (MW) from Natural Gas,Net Generation (MW) from Nuclear,Net Generation (MW) from All Petroleum Products,Net Generation (MW) from Hydropower Excluding Pumped Storage,Net Generation (MW) from Pumped Storage,Net Generation (MW) from Solar without Integrated Battery Storage,Net Generation (MW) from Solar with Integrated Battery Storage,Net Generation (MW) from Wind without Integrated Battery Storage,Net Generation (MW) from Wind with Integrated Battery Storage,Net Generation (MW) from Battery Storage,Net Generation (MW) from Other Energy Storage,Net Generation (MW) from Unknown Energy Storage,Net Generation (MW) from Geothermal,Net Generation (MW) from Other Fuel Sources,Net Generation (MW) from Unknown Fuel Sources,Net Generation (MW) from Coal (Imputed),Net Generation (MW) from Natural Gas (Imputed),Net Generation (MW) from Nuclear (Imputed),Net Generation (MW) from All Petroleum Products (Imputed),Net Generation (MW) from Hydropower Excluding Pumped Storage (Imputed),Net Generation (MW) from Pumped Storage (Imputed),Net Generation (MW) from Solar without Integrated Battery Storage (Imputed),Net Generation (MW) from Solar with Integrated Battery Storage (Imputed),Net Generation (MW) from Wind without Integrated Battery Storage (Imputed),Net Generation (MW) from Wind with Integrated Battery Storage (Imputed),Net Generation (MW) from Battery Storage (Imputed),Net Generation (MW) from Other Energy Storage (Imputed),Net Generation (MW) from Unknown Energy Storage (Imputed),Net Generation (MW) from Geothermal (Imputed),Net Generation (MW) from Other Fuel Sources (Imputed),Net Generation (MW) from Unknown Fuel Sources (Imputed),Net Generation (MW) from Coal (Adjusted),Net Generation (MW) from Natural Gas (Adjusted),Net Generation (MW) from Nuclear (Adjusted),Net Generation (MW) from All Petroleum Products (Adjusted),Net Generation (MW) from Hydropower Excluding Pumped Storage (Adjusted),Net Generation (MW) from Pumped Storage (Adjusted),Net Generation (MW) from Solar without Integrated Battery Storage (Adjusted),Net Generation (MW) from Solar witho Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind without Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind with Integrated Battery Storage (Adjusted),Net Generation (MW) from Battery Storage (Adjusted),Net Generation (MW) from Other Energy Storage (Adjusted),Net Generation (MW) from Unknown Energy Storage (Adjusted),Net Generation (MW) from Geothermal (Adjusted),Net Generation (MW) from Other Fuel Sources (Adjusted),Net Generation (MW) from Unknown Fuel Sources (Adjusted),Region
113873,ISNE,11/21/2025,19,11/21/2025 7:00:00 PM,11/22/2025 12:00:00 AM,15120.0,15195.0,15164.0,-31.0,-31.0,NaN,NaN,NaN,15195.0,15164.0,-31.0,3.0,9456.0,3297.0,213.0,571.0,350.0,1.0,0.0,629.0,NaN,70.0,NaN,NaN,NaN,574.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,9456.0,3297.0,213.0,571.0,350.0,1.0,0.0,629.0,NaN,70.0,NaN,NaN,NaN,574.0,NaN,NE
31461,CISO,07/23/2025,16,07/23/2025 4:00:00 PM,07/23/2025 11:00:00 PM,25934.0,28201.0,29963.0,721.0,721.0,NaN,NaN,NaN,28201.0,29963.0,721.0,0.0,9787.0,2262.0,-3.0,1065.0,NaN,17544.0,NaN,2387.0,NaN,NaN,NaN,NaN,NaN,-3079.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,9787.0,2262.0,-3.0,1065.0,NaN,17544.0,NaN,2387.0,NaN,NaN,NaN,NaN,NaN,-3079.0,NaN,CAL
259271,WAUW,11/06/2025,23,11/06/2025 11:00:00 PM,11/07/2025 6:00:00 AM,71.0,77.0,53.0,-24.0,-24.0,NaN,NaN,NaN,77.0,53.0,-24.0,NaN,NaN,NaN,NaN,53.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [52]:
balance.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Balancing Authority,267218,61,AECI,4417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Date,267218,184,11/02/2025,1490,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour Number,267218.0,NaN,NaN,NaN,12.502339,6.923663,1.0,7.0,13.0,19.0,25.0
Local Time at End of Hour,267218,4416,11/02/2025 1:00:00 AM,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UTC Time at End of Hour,267218,4420,07/01/2025 8:00:00 AM,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Demand Forecast (MW),233431.0,NaN,NaN,NaN,9465.17759,19303.64298,-227966.0,791.0,2449.0,6493.0,156738.0
Demand (MW),233639.0,NaN,NaN,NaN,9503.464246,19220.239208,-39388.0,815.0,2544.0,6665.0,810948.0
Net Generation (MW),266717.0,NaN,NaN,NaN,8304.861227,18525.972929,-2576.0,547.0,1759.0,5935.0,810705.0
Total Interchange (MW),266739.0,NaN,NaN,NaN,-78.283611,1344.438016,-11031.0,-496.0,-8.0,387.0,40651.0
Sum(Valid DIBAs) (MW),265658.0,NaN,NaN,NaN,-60.813783,1416.714082,-57015.0,-505.0,-13.0,374.0,40651.0


### EIA-930 Subregion Data

In [53]:
subregion = pd.read_csv(DATA_DIR / "eia930" / "EIA930_SUBREGION_2025_Jul_Dec.csv")
print("Rows and columns:", subregion.shape)
subregion.sample(3)

Rows and columns: (366596, 7)


,Balancing Authority,Data Date,Hour Number,Sub-Region,Demand (MW),Local Time at End of Hour,UTC Time at End of Hour
52146,ERCO,12/27/2025,13,SOUT,4597.0,12/27/2025 1:00:00 PM,12/27/2025 7:00:00 PM
1081,CISO,07/12/2025,7,SCE,10835.0,07/12/2025 7:00:00 AM,07/12/2025 2:00:00 PM
82281,ISNE,11/30/2025,11,4006,1667.0,11/30/2025 11:00:00 AM,11/30/2025 4:00:00 PM


In [54]:
subregion.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Balancing Authority,366596,8,PJM,88340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Date,366596,184,11/02/2025,2060,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour Number,366596.0,NaN,NaN,NaN,12.502319,6.923647,1.0,7.0,13.0,19.0,25.0
Sub-Region,366596,83,PGAE,4417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Demand (MW),365441.0,NaN,NaN,NaN,3942.233748,4968.20665,-4.0,814.0,1889.0,4780.0,32436.0
Local Time at End of Hour,366596,4416,11/02/2025 1:00:00 AM,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UTC Time at End of Hour,366596,4420,07/01/2025 8:00:00 AM,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save cleaned data

Save the cleaned data centers, including the `is_crypto` label, to `data/processed/`. This file is the input to `03-supervised-learning.ipynb`.

In [55]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")

Saved: fractracker_data_centers_clean.csv with 1720 rows and 101 columns


### Join Energy on cleaned fractacker data

In [56]:
data_centers_prep = data_centers.copy().rename(columns={"state":"State", "county":"County"})
data_center_energy_merge = data_centers_prep.merge(pivot_sales, on="State", how="left")
data_center_with_energy = data_center_energy_merge.merge(group_by_stat, on=["State","County"], how="left")
data_center_with_energy = data_center_with_energy.rename(columns={"State":"state","County":"county"}).reset_index()
data_center_with_energy.sample(5)

,index,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup,crypto_idx,date,name,address_crypto,city_crypto,state_crypto,lat_crypto,long_crypto,company,probable_power_source,mw_nytimes,nearest_dc_miles,nearest_dc_name,nearest_dc_operator,zip_crypto,atlas_idx,name_atlas,operator,facility_type,lat_atlas,long_atlas,capacity_operational_mw,capacity_planned_mw,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind,buildings_low,buildings_high,air,hybrid_air_water,water,closed_loop,fans,open_loop,project_cost_low,project_cost_high,is_crypto,summer_Thousand Dollars.1,winter_Thousand Dollars.1,summer_Megawatthours.1,winter_Megawatthours.1,summer_Count.1,winter_Count.1,summer_Cents/kWh.1,winter_Cents/kWh.1,generator_count,generator_summer_max,generator_winter_max,generator_capacity_summer,generator_capacity_winter,generator_avg_time_to_full,All Other,Batteries,Coal Integrated Gasification Combined Cycle,Conventional Hydroelectric,Conventional Steam Coal,Flywheels,Geothermal,Hydroelectric Pumped Storage,Landfill Gas,Municipal Solid Waste,Natural Gas Fired Combined Cycle,Natural Gas Fired Combustion Turbine,Natural Gas Internal Combustion Engine,Natural Gas Steam Turbine,Natural Gas with Compressed Air Storage,Nuclear,Offshore Wind Turbine,Onshore Wind Turbine,Other Gases,Other Natural Gas,Other Waste Biomass,Petroleum Coke,Petroleum Liquids,Solar Photovoltaic,Solar Thermal with Energy Storage,Solar Thermal without Energy Storage,Wood/Wood Waste Biomass,Capacity_Sum
1117,1117,3974540,16f0afdc-94e7-4360-93dc-5c0bf374a842,1768854703438-7754,Riot Platforms Rockdale Data Center Campus,2721 charles martin hall rd,Rockdale,TX,76567.0,Milam,30.573387,-97.07707,Expanding,High,Google Maps only,HPC/AI,Riot Platforms,AMD,700,700.0,700.0,Hyperscale (100-999 MW),5,Grid (unspecified mix),NaN,NaN,8,NaN,NaN,40000.0,200.0,$75 million,2027,Unknown,NaN,Unknown,NaN,NaN,NaN,NaN,Bitcoin operator expanding to HPC. 8 buildings...,KE/LB,Media Monitoring,https://www.tdlr.texas.gov/TABS/Projects/TABS2...,https://www.datacenterdynamics.com/en/news/amd...,https://www.riotplatforms.com/locations/rockdale/,https://finance.yahoo.com/markets/stocks/artic...,NaN,NaN,NaN,NaN,1.768781e+12,1.784246e+12,NaN,248.0,7/24/2025,Riot Rockdale,2721 charles martin hall rd,Rockdale,TX,30.56359,-97.08498,Riot Platforms,Grid,700 MW,0.790603,Bitdeer AI,Bitdeer,76567.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,0,0,0,0,0,8.0,8.0,0,0,0,0,0,0,75000000.0,75000000.0,1,1.332645e+06,1.171187e+06,1.531069e+07,1.365712e+07,1.537500e+06,1.537970e+06,8.696667,8.581667,6.0,250.0,250.0,990.0,990.0,140.700000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1980.0
492,492,3973915,7e186215-6e45-405f-85bb-84678ec46fcb,299,Prosperity Drive Data Center,12401 prosperity dr,Silver Spring,MD,20904.0,Montgomery,39.058590,-76.96440,Operating,High,Google Maps and other sources,data_center,NaN,NaN,5-100,5.0,100.0,Unknown,1,NaN,NaN,NaN,NaN,NaN,NaN,214000.0,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://www.datacenterdynamics.com/en/news/dat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.755562e+12,1.755562e+12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1972.0,12401 Prosperity Drive D

In [57]:
data_center_with_energy.to_csv("data/processed/fractracker_data_centers_with_eia_cleaned.csv", index=False)
test = pd.read_csv("data/processed/fractracker_data_centers_with_eia_cleaned.csv", header=0)

In [58]:
test.head(5)

,index,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup,crypto_idx,date,name,address_crypto,city_crypto,state_crypto,lat_crypto,long_crypto,company,probable_power_source,mw_nytimes,nearest_dc_miles,nearest_dc_name,nearest_dc_operator,zip_crypto,atlas_idx,name_atlas,operator,facility_type,lat_atlas,long_atlas,capacity_operational_mw,capacity_planned_mw,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind,buildings_low,buildings_high,air,hybrid_air_water,water,closed_loop,fans,open_loop,project_cost_low,project_cost_high,is_crypto,summer_Thousand Dollars.1,winter_Thousand Dollars.1,summer_Megawatthours.1,winter_Megawatthours.1,summer_Count.1,winter_Count.1,summer_Cents/kWh.1,winter_Cents/kWh.1,generator_count,generator_summer_max,generator_winter_max,generator_capacity_summer,generator_capacity_winter,generator_avg_time_to_full,All Other,Batteries,Coal Integrated Gasification Combined Cycle,Conventional Hydroelectric,Conventional Steam Coal,Flywheels,Geothermal,Hydroelectric Pumped Storage,Landfill Gas,Municipal Solid Waste,Natural Gas Fired Combined Cycle,Natural Gas Fired Combustion Turbine,Natural Gas Internal Combustion Engine,Natural Gas Steam Turbine,Natural Gas with Compressed Air Storage,Nuclear,Offshore Wind Turbine,Onshore Wind Turbine,Other Gases,Other Natural Gas,Other Waste Biomass,Petroleum Coke,Petroleum Liquids,Solar Photovoltaic,Solar Thermal with Energy Storage,Solar Thermal without Energy Storage,Wood/Wood Waste Biomass,Capacity_Sum
0,0,3973423,3d8a6ce3-5e4b-4ede-b9ca-c9ab4f5dfa9b,1779305646902-5353,Stak Energy Data Center,dalton hwy 26 miles s of deadhorse,North Slope Borough,AK,99734.0,North Slope,69.90071,-148.81477,Proposed,Medium,Google Maps and other sources,AI/Cloud Computing,Stak,NaN,3000,3000.0,3000.0,"Mega campus (>1,000 MW)",6,Natural gas,Yes,NaN,NaN,Air,NaN,NaN,715.0,$500 million,2028,Unknown,NaN,Emerging Concern,NaN,NaN,NaN,NaN,NaN,KE/JH,Media Monitoring,https://www.datacenterdynamics.com/en/news/sta...,https://www.cbc.ca/news/canada/north/yukon-ala...,https://alaskapublic.org/news/economy/energy/2...,https://alaskabeacon.com/2026/07/06/data-cente...,https://aws.state.ak.us/OnlinePublicNotices/No...,NaN,NaN,NaN,1.779235e+12,1.783469e+12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,1,0,0,0,0,0,NaN,NaN,1,0,0,0,0,0,5.000000e+08,5.000000e+08,0,48187.756667,50552.120000,2.120679e+05,2.303088e+05,58400.833333,58067.333333,22.718333,21.958333,48.0,6.9,8.0,69.0,75.8,503.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,144.8
1,1,3973424,c65854df-23a2-4d76-8ab7-c24323cb63f9,1779306105674-8821,Prudhoe Bay Data Center,dalton hwy,Prudhoe Bay,AK,99734.0,North Slope,70.18478,-148.44000,Operating,Medium,Google Maps and other sources,NaN,"Far North Digital, LLC",NaN,120,120.0,120.0,Hyperscale (100-999 MW),5,NaN,NaN,NaN,NaN,Air,NaN,NaN,100.0,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://www.fn-digital.com/data-center,https://www.datacenterdynamics.com/en/news/sta...,NaN,NaN,NaN,NaN,NaN,NaN,1.779235e+12,1.779235e+12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0